# Export AGC settings and nearest minispect data for raw example frames

This notebook locates each completely raw target frame within naturally ordered world-camera chunks, retrieves every AGC setting except timestamp, finds the nearest minispect sample on the shared clock, and saves the result beside the example TIFF. Raw chunks are examined one at a time to limit memory use. Frame matching is exact: the target and stored raw frame must have identical shapes and pixel values.

In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import numpy as np
import cv2
from natsort import natsorted
from scipy.io import savemat


def locate_project_root(start: Path | None = None) -> Path:
    """Locate the lightLoggerAnalysis repository from a notebook launch path."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "code").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the lightLoggerAnalysis repository.")


PROJECT_ROOT = locate_project_root()
CHUNK_IO_PATH = PROJECT_ROOT / "code" / "library" / "matlabIO" / "python_libraries"

if str(CHUNK_IO_PATH) not in sys.path:
    sys.path.insert(0, str(CHUNK_IO_PATH))

import chunk_io

chunk_io = importlib.reload(chunk_io)

In [ ]:
EXAMPLE_FRAMES_DIR = PROJECT_ROOT / "data" / "exampleWorldCameraImages"


def world_timestamp_at_global_frame_index(
    raw_chunks_path: str, global_frame_index: int
) -> float:
    """Map a raw frame index to its world timestamp in seconds."""
    if global_frame_index < 0:
        raise ValueError("global_frame_index must be nonnegative.")

    # The global index returned by find_frame_index counts only frames that
    # physically occur in the naturally ordered raw world chunks. Walk those
    # same chunks and translate the global index to one local chunk index.
    global_offset = 0
    for metadata_path, value_path in chunk_io.group_sensors_files(raw_chunks_path)["W"]:
        metadata = np.load(metadata_path, mmap_mode="r")
        values = np.load(value_path, mmap_mode="r")
        if len(metadata) != len(values):
            raise ValueError(
                f"World metadata/value length mismatch: {metadata_path}, {value_path}"
            )

        chunk_stop = global_offset + len(values)
        if global_frame_index < chunk_stop:
            local_index = global_frame_index - global_offset
            timestamp_nanoseconds = (
                metadata[local_index]
                if metadata.ndim == 1
                else metadata[local_index, 0]
            )
            # Raw world timestamps are nanoseconds; find_nearest_neighbor
            # accepts and returns timestamps in seconds.
            return float(timestamp_nanoseconds) / 1e9
        global_offset = chunk_stop

    raise IndexError(
        f"Global frame index {global_frame_index} exceeds {global_offset} raw frames."
    )


def find_example_frame_context(
    raw_chunks_path: str,
    target: np.ndarray,
    include_minispect: bool = True,
    verbose: bool = True,
) -> dict | None:
    """Return matched world context and the nearest minispect entry.

    Args:
        raw_chunks_path: Directory containing the raw recording chunks.
        target: Raw target frame. Its shape must exactly match the stored
            frames in the raw world chunks.
        include_minispect: Whether to retrieve the nearest minispect entry.
        verbose: Whether to display chunk-level search progress.

    Returns:
        Dictionary containing the global raw-frame index and `W` / optional
        `M` nearest-neighbor entries. `W["AGCSettings"]` contains every
        world metadata setting except timestamp. Returns `None` when the
        target does not occur in the raw chunks.
    """

    target = np.asarray(target)
    if target.ndim not in (2, 3):
        raise ValueError("target must be a two-dimensional grayscale or three-dimensional RGB frame.")

    matching_frame_index = chunk_io.find_frame_index(
        raw_chunks_path,
        target,
        verbose=verbose,
    )
    if matching_frame_index is None:
        return None

    # Use that exact raw frame index to retrieve its timestamp directly from
    # the aligned world metadata row, without inserting synthetic gap rows.
    timestamp_seconds = world_timestamp_at_global_frame_index(
        raw_chunks_path, matching_frame_index
    )

    # At the target world timestamp, retrieve the complete world row (frame,
    # timestamp, and AGC settings) and the nearest minispect packet.
    requested_sensors = ("W", "M") if include_minispect else "W"
    nearest = chunk_io.find_nearest_neighbor(
        raw_chunks_path, timestamp_seconds, requested_sensors
    )

    # These checks ensure the index-to-timestamp mapping returned us to the
    # exact target world frame before anything is written to disk.
    if nearest["W"]["timestamp"] != timestamp_seconds:
        raise ValueError("Nearest world timestamp does not match the target timestamp.")
    if not np.array_equal(nearest["W"]["value"], target):
        raise ValueError("Nearest world entry does not match the target frame.")
    return {"globalFrameIndex": matching_frame_index, **nearest}


def export_example_frame_context(
    image_path: Path,
    raw_chunks_path: str,
    include_minispect: bool = True,
) -> Path:
    """Save AGC settings and nearest sensor entries beside an example TIFF."""
    target = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
    if target is None:
        raise FileNotFoundError(image_path)
    context = find_example_frame_context(
        raw_chunks_path, target, include_minispect=include_minispect
    )
    if context is None:
        raise ValueError(f"Could not locate target frame: {image_path}")

    condition, sequence_text = image_path.stem.rsplit("_", 1)
    output_label = "AGCandMS" if include_minispect else "AGC"
    output_path = image_path.with_name(
        f"{condition}_{output_label}_{int(sequence_text):02d}.mat"
    )
    # Use descriptive MATLAB variable names instead of serializing the
    # internal W/M lookup dictionary verbatim.
    matlab_payload = {
        "sourceImageFilename": image_path.name,
        "globalWorldFrameIndex": np.int64(context["globalFrameIndex"]),
        "worldTimestampSeconds": context["W"]["timestamp"],
        "worldFrame": context["W"]["value"],
        "AGCSettings": context["W"]["AGCSettings"],
    }
    if include_minispect:
        matlab_payload["minispectTimestampSeconds"] = context["M"]["timestamp"]
        matlab_payload["minispectValue"] = context["M"]["value"]
    savemat(output_path, matlab_payload, do_compression=True, long_field_names=True)

    print(f"Saved {output_path.name}")
    print("AGCSettings:", context["W"]["AGCSettings"])
    return output_path

In [ ]:
indoor_raw_path: str = "/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA"

for image_path in natsorted(EXAMPLE_FRAMES_DIR.glob("indoor_*.tiff")):
    export_example_frame_context(image_path, indoor_raw_path)

In [ ]:
outdoor_raw_path: str = "/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkOutdoor/GKA"

for image_path in natsorted(EXAMPLE_FRAMES_DIR.glob("outdoor_*.tiff")):
    export_example_frame_context(image_path, outdoor_raw_path)

In [ ]:
planetarium_raw: str = "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/flatFieldingFunction/planetarium_fielding_function_raw"

for image_path in natsorted(EXAMPLE_FRAMES_DIR.glob("planetarium_*.tiff")):
    # The planetarium calibration recording contains world-camera data only.
    export_example_frame_context(image_path, planetarium_raw, include_minispect=False)

## Usage

Pass an example TIFF and its source recording to the export helper:

```python
output_path = export_example_frame_context(
    EXAMPLE_FRAMES_DIR / "indoor_1.tiff",
    raw_chunks_path="/path/to/recording/GKA",
)
```

For example, `indoor_1.tiff` produces `indoor_AGCandMS_01.mat`. The MAT file contains `sourceImageFilename`, `globalWorldFrameIndex`, `worldTimestampSeconds`, `worldFrame`, `AGCSettings`, `minispectTimestampSeconds`, and `minispectValue`. `minispectValue` is a MATLAB struct containing the parsed `AS`, `TS`, `LS`, and `TEMP` arrays. A world-only recording produces an `_AGC_` file without the two minispect variables.